# Data Wrangling

Load & transform NBA dataset

Ensure the two datasets are located in folder called data (or update load filepath)

### Import & Load Data

In [27]:
# Imports
import pandas as pd
import polars as pl
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder


In [58]:
# Load raw csv data
df_shots = pl.read_csv("../data/shotdetail_2024.csv", try_parse_dates=True)
df_cdnnba = pl.read_csv("../data/cdnnba_2024.csv", try_parse_dates=True)

# Merge cdnnba & shots dataframes
df_merged = df_shots.join(df_cdnnba, how='left', left_on=['GAME_ID', 'GAME_EVENT_ID'], right_on=['gameId', 'actionNumber'])
df_merged = df_merged.with_columns(pl.concat_str([pl.col('GAME_ID'), pl.col('GAME_EVENT_ID')], separator="_").alias('Record_Id'))


### Exploration

In [23]:
df_merged.dtypes

[String,
 Int64,
 Int64,
 Int64,
 String,
 Int64,
 String,
 Int64,
 Int64,
 Int64,
 String,
 String,
 String,
 String,
 String,
 String,
 Int64,
 Int64,
 Int64,
 Int64,
 Int64,
 Int64,
 String,
 String,
 String,
 Datetime(time_unit='us', time_zone='UTC'),
 Int64,
 String,
 String,
 String,
 String,
 Int64,
 Float64,
 Float64,
 Int64,
 Int64,
 Int64,
 Datetime(time_unit='us', time_zone='UTC'),
 Int64,
 Boolean,
 Int64,
 Int64,
 Int64,
 String,
 String,
 String,
 Int64,
 String,
 String,
 String,
 Int64,
 String,
 String,
 String,
 Int64,
 String,
 Int64,
 String,
 String,
 Float64,
 String,
 String,
 Int64,
 Int64,
 Int64,
 Int64,
 Int64,
 Int64,
 Int64,
 String,
 Int64,
 Int64,
 String,
 Int64,
 Int64,
 Int64,
 Int64,
 String,
 Int64,
 String]

In [24]:
df_merged.describe()

statistic,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,possession,scoreHome,…,side,description,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,shotDistance,shotResult,blockPlayerName,blockPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,turnoverTotal,stealPlayerName,stealPersonId,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,Record_Id
str,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,str,str,str,f64,f64,f64,f64,f64,…,str,str,str,f64,str,str,str,f64,str,str,str,f64,str,f64,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,f64,f64,f64,f64,str,f64,str
"""count""","""219527""",219527.0,219527.0,219527.0,"""219527""",219527.0,"""219527""",219527.0,219527.0,219527.0,"""219527""","""219527""","""219527""","""219527""","""219527""","""219527""",219527.0,219527.0,219527.0,219527.0,219527.0,219527.0,"""219527""","""219527""","""219524""","""219524""",219524.0,"""219524""","""219524""","""219524""","""219524""",219524.0,219524.0,219524.0,219524.0,219524.0,…,"""219524""","""219524""","""219524""",219524.0,"""219524""","""150690""","""0""",0.0,"""219524""","""219524""","""0""",0.0,"""0""",0.0,"""219523""","""219524""",219524.0,"""219524""","""11993""",11993.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,102566.0,"""65312""",65312.0,65312.0,0.0,0.0,"""0""",0.0,"""219527"""
"""null_count""","""0""",0.0,0.0,0.0,"""0""",0.0,"""0""",0.0,0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""3""","""3""",3.0,"""3""","""3""","""3""","""3""",3.0,3.0,3.0,3.0,3.0,…,"""3""","""3""","""3""",3.0,"""3""","""68837""","""219527""",219527.0,"""3""","""3""","""219527""",219527.0,"""219527""",219527.0,"""4""","""3""",3.0,"""3""","""207534""",207534.0,219527.0,219527.0,219527.0,219527.0,219527.0,219527.0,"""219527""",219527.0,116961.0,"""154215""",154215.0,154215.0,219527.0,219527.0,"""219527""",219527.0,"""0"""
"""mean""",null,2.2401e7,330.567493,1.3752e6,null,1.6106e9,null,2.480843,5.385274,28.853617,null,null,null,null,null,null,14.003635,-2.252251,97.446464,1.0,0.467214,2.0247e7,null,null,null,"""2025-01-19 00:54:21.708397+00:…",2.480836,null,null,null,null,1.3752e6,49.833238,49.493792,1.6106e9,56.89305,…,null,null,null,1.6106e9,null,null,null,null,null,null,null,null,null,null,null,null,14.484858,null,null,1.3805e6,null,null,null,null,null,null,null,null,9.854425,null,1.3309e6,3.121279,null,null,null,null,null
"""std""",null,354.580103,199.058742,550394.782499,null,8.705154,null,1.127515,3.450603,17.407554,null,null,null,null,null,null,10.698021,116.722689,96.627021,0.0,0.498925,4443.34033,null,null,null,null,1.127517,null,null,null,null,550386.873113,35.563115,23.342663,8.70513,34.286788,…,null,null,null,8.70513,null,null,null,null,null,null,null,null,null,null,null,null,10.714503,null,null,545722.906185,null,null,null,null,null,null,null,null,7.306994,null,585964.051679,2.412323,null,null,null,null,null
"""min""","""Shot Chart Detail""",2.2400001e7,4.0,2544.0,"""A.J. Lawson""",1.6106e9,"""Atlanta Hawks""",1.0,0.0,0.0,"""Made Shot""","""Alley Oop Dunk Shot""","""2PT Field Goal""","""Above the Break 3""","""Back Court(BC)""","""16-24 ft.""",0.0,-250.0,-52.0,1.0,0.0,2.0241022e7,"""ATL""","""ATL""","""PT00M00.00S""","""2024-10-22 23:36:07.500000+00:…",1.0,"""OVERTIME""","""2pt""","""DUNK""","""""",2544.0,0.0,0.0,1.6106e9,0.0,…,"""left""","""

## Data Cleaning

In [ ]:
# Convert data types
df_merged = df_merged.with_columns(
    pl.col('GAME_DATE').cast(pl.String).str.to_datetime(format="%Y%m%d").alias('game_date'),
    pl.when(pl.col('HTM') == pl.col('teamTricode')).then(1).otherwise(0).alias('isHomeTeam'),
    pl.duration(minutes=pl.col('MINUTES_REMAINING'), seconds=pl.col('SECONDS_REMAINING')).alias('time_remaining'),
    (pl.col('scoreHome') - pl.col('scoreAway')).alias('score_margin_after'),
    pl.when(pl.col('SHOT_MADE_FLAG') == 1).then(
        pl.when(pl.col('actionType') == "3pt").then(pl.lit(3)).otherwise(pl.lit(2))
        ).otherwise(pl.lit(0)).alias('score_change'), 
    pl.when(pl.col('shotDistance').is_null()).then(pl.col('SHOT_DISTANCE')).otherwise(pl.col('shotDistance')).alias('shotDistance')
)

# Convert score to be relative to player taking actions perspective
df_merged = df_merged.with_columns(
    pl.when(pl.col('teamTricode') == pl.col('HTM'))
        .then(pl.col('score_margin_after') - pl.col('score_change'))
    .otherwise(pl.col('score_margin_after')*-1 - pl.col('score_change')).alias('relative_margin_before')
)

In [47]:
df_merged.head(5)

GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,possession,scoreHome,scoreAway,…,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,shotDistance,shotResult,blockPlayerName,blockPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,turnoverTotal,stealPlayerName,stealPersonId,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,Record_Id,game_date,atHome,time_remaining,score_margin_after,score_change,relative_margin_before
str,i64,i64,i64,str,i64,str,i64,i64,i64,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str,str,str,"datetime[μs, UTC]",i64,str,str,str,str,i64,f64,f64,i64,i64,i64,…,str,i64,str,str,str,i64,str,i64,str,str,f64,str,str,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,str,i64,i64,i64,i64,str,i64,str,datetime[μs],i32,duration[μs],i64,i32,i64
"""Shot Chart Detail""",22400001,7,1642258,"""Zaccharie Risacher""",1610612737,"""Atlanta Hawks""",1,11,43,"""Missed Shot""","""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Left Side Center(LC)""","""24+ ft.""",26,-168,205,1,0,20241112,"""BOS""","""ATL""","""PT11M43.00S""",2024-11-13 00:10:40 UTC,1,"""REGULAR""","""3pt""","""Jump Shot""","""""",1642258,27.414586,83.578431,1610612737,0,0,…,null,null,"""Risacher""","""Z. Risacher""",null,null,null,null,"""Above the Break 3""","""24+ Left Center""",26.51,"""Missed""","""Tatum""",1628369,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""22400001_7""",2024-11-12 00:00:00,0,11m 43s,0,0,0
"""Shot Chart Detail""",22400001,10,1630552,"""Jalen Johnson""",1610612737,"""Atlanta Hawks""",1,11,38,"""Missed Shot""","""Driving Floating Bank Jump Sho…","""2PT Field Goal""","""Mid-Range""","""Left Side(L)""","""8-16 ft.""",13,-136,-1,1,0,20241112,"""BOS""","""ATL""","""PT11M38.00S""",2024-11-13 00:10:46 UTC,1,"""REGULAR""","""2pt""","""Jump Shot""","""2ndchance""",1630552,5.469777,77.205882,1610612737,0,0,…,null,null,"""Johnson""","""J. Johnson""",null,null,null,null,"""Mid-Range""","""8-16 Left""",13.6,"""Missed""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""22400001_10""",2024-11-12 00:00:00,0,11m 38s,0,0,0
"""Shot Chart Detail""",22400001,21,1630552,"""Jalen Johnson""",1610612737,"""Atlanta Hawks""",1,10,50,"""Made Shot""","""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Right Side Center(RC)""","""24+ ft.""",25,157,203,1,1,20241112,"""BOS""","""ATL""","""PT10M50.00S""",2024-11-13 00:11:45.500 UTC,1,"""REGULAR""","""3pt""","""Jump Shot""","""fromturnover""",1630552,27.151774,18.627451,1610612737,0,3,…,null,null,"""Johnson""","""J. Johnson""",null,null,null,null,"""Above the Break 3""","""24+ Right Center""",25.63,"""Made""",null,null,null,null,null,null,null,null,null,null,3,"""K. Wallace""",1630811,1,null,null,null,null,"""22400001_21""",2024-11-12 00:00:00,0,10m 50s,-3,3,0
"""Shot Chart Detail""",22400001,34,1630811,"""Keaton Wallace""",1610612737,"""Atlanta Hawks""",1,9,47,"""Missed Shot""","""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Left Side Center(LC)""","""24+ ft.""",25,-176,184,1,0,20241112,"""BOS""","""ATL""","""PT09M47.00S""",2024-11-13 00:13:42.600 UTC,1,"""REGULAR""","""3pt""","""Jump Shot""","""""",1630811,25.180683,85.294118,1610612737,3,3,…,null,null,"""Wallace""","""K. Wallace""",null,null,null,null,"""Above the Break 3""","""24+ Left Center""",25.51,"""Missed""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""22400001_

In [12]:
# Confirm no mismatching data between event types & shot flags
made_shots_mismatch = df_merged.filter(
    (pl.col('EVENT_TYPE') =='Made Shot') & (pl.col('SHOT_ATTEMPTED_FLAG')!=1) & (pl.col('SHOT_MADE_FLAG')!=1)
)

missed_shots_mismatch = df_merged.filter(
    (pl.col('EVENT_TYPE') =='Missed Shot') & (pl.col('SHOT_ATTEMPTED_FLAG')!=1) & (pl.col('SHOT_MADE_FLAG')!=0)
)

In [52]:
# Drop redundant (i.e. duplicate) columns
df_combined = df_merged.drop(['period', 'personId', 'actionType', 'GAME_DATE', 'SHOT_DISTANCE', 'teamId'])

# Drop columns deemed not required
df_combined = df_combined.drop([
    'EVENT_TYPE',
    'MINUTES_REMAINING',
    'SECONDS_REMAINING',
    'foulPersonalTotal',
    'foulTechnicalTotal',
    'foulDrawnPlayerName',
    'foulDrawnPersonId',
    'area',
    'areaDetail',
    'shotResult',
    'officialId',
    'turnoverTotal',
    'stealPlayerName',
    'stealPersonId',
    'reboundTotal',
    'reboundDefensiveTotal',
    'reboundOffensiveTotal',
    'blockPlayerName',
    'blockPersonId',
    'jumpBallLostPlayerName',
    'jumpBallLostPersonId',
    'jumpBallWonPlayerName',
    'jumpBallWonPersonId', 
    'jumpBallRecoveredName',    
    'jumpBallRecoverdPersonId',
    'shotActionNumber', 
    'SHOT_ATTEMPTED_FLAG', 
    'playerName',
    'playerNameI', 
    'teamTricode', 
    'possession',
    'xLegacy',
    'yLegacy',
    'x',
    'y',
    'description',
    'personIdsFilter',
    'descriptor'
])
df_combined

GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,LOC_X,LOC_Y,SHOT_MADE_FLAG,HTM,VTM,clock,timeActual,periodType,subType,qualifiers,scoreHome,scoreAway,edited,orderNumber,isTargetScoreLastPeriod,isFieldGoal,side,shotDistance,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,Record_Id,game_date,atHome,time_remaining,score_margin_after,score_change,relative_margin_before,isHomeTeam
str,i64,i64,i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str,str,"datetime[μs, UTC]",str,str,str,i64,i64,"datetime[μs, UTC]",i64,bool,i64,str,f64,i64,str,i64,i64,str,datetime[μs],i32,duration[μs],i64,i32,i64,i32
"""Shot Chart Detail""",22400001,7,1642258,"""Zaccharie Risacher""",1610612737,"""Atlanta Hawks""",1,"""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Left Side Center(LC)""","""24+ ft.""",-168,205,0,"""BOS""","""ATL""","""PT11M43.00S""",2024-11-13 00:10:40 UTC,"""REGULAR""","""Jump Shot""","""""",0,0,2024-11-13 00:10:47 UTC,70000,false,1,"""left""",26.51,null,null,null,null,"""22400001_7""",2024-11-12 00:00:00,0,11m 43s,0,0,0,0
"""Shot Chart Detail""",22400001,10,1630552,"""Jalen Johnson""",1610612737,"""Atlanta Hawks""",1,"""Driving Floating Bank Jump Sho…","""2PT Field Goal""","""Mid-Range""","""Left Side(L)""","""8-16 ft.""",-136,-1,0,"""BOS""","""ATL""","""PT11M38.00S""",2024-11-13 00:10:46 UTC,"""REGULAR""","""Jump Shot""","""2ndchance""",0,0,2024-11-13 00:11:05 UTC,100000,false,1,"""left""",13.6,null,null,null,null,"""22400001_10""",2024-11-12 00:00:00,0,11m 38s,0,0,0,0
"""Shot Chart Detail""",22400001,21,1630552,"""Jalen Johnson""",1610612737,"""Atlanta Hawks""",1,"""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Right Side Center(RC)""","""24+ ft.""",157,203,1,"""BOS""","""ATL""","""PT10M50.00S""",2024-11-13 00:11:45.500 UTC,"""REGULAR""","""Jump Shot""","""fromturnover""",0,3,2024-11-13 00:11:50 UTC,210000,false,1,"""left""",25.63,3,"""K. Wallace""",1630811,1,"""22400001_21""",2024-11-12 00:00:00,0,10m 50s,-3,3,0,0
"""Shot Chart Detail""",22400001,34,1630811,"""Keaton Wallace""",1610612737,"""Atlanta Hawks""",1,"""Jump Shot""","""3PT Field Goal""","""Above the Break 3""","""Left Side Center(LC)""","""24+ ft.""",-176,184,0,"""BOS""","""ATL""","""PT09M47.00S""",2024-11-13 00:13:42.600 UTC,"""REGULAR""","""Jump Shot""","""""",3,3,2024-11-13 00:13:52 UTC,330000,false,1,"""left""",25.51,null,null,null,null,"""22400001_34""",2024-11-12 00:00:00,0,9m 47s,0,0,0,0
"""Shot Chart Detail""",22400001,36,203991,"""Clint Capela""",1610612737,"""Atlanta Hawks""",1,"""Putback Layup Shot""","""2PT Field Goal""","""Restricted Area""","""Center(C)""","""Less Than 8 ft.""",-25,8,1,"""BOS""","""ATL""","""PT09M44.00S""",2024-11-13 00:13:47.100 UTC,"""REGULAR""","""Layup""","""pointsinthepaint, 2ndchance""",3,5,2024-11-13 00:13:57 UTC,350000,false,1,"""left""",2.62,2,null,null,null,"""22400001_36""",2024-11-12 00:00:00,0,9m 44s,-2,2,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Shot Chart Detail""",22401217,643,1629673,"""Jordan Poole""",1610612764,"""Washington Wizards""",4,"""Driving Layup Shot""","""2PT Field Goal""","""Restricted Area""","""Center(C)""","""Less Than 8 ft.""",15,1,1,"""WAS""","""BOS""","""PT03M14.00S""",2024-12-16 01:10:35.800 UTC,"""REGULAR""","""Layup""","""pointsinthepaint""",90,105,2024-12-16 01:10:41 UTC,6200000,false,1,"""left""",1.48,18,null,null,null,"""22401217_643""",2024-12-15 00:00:00,1,3m 14s,-15,2,-17,1
"""Shot Chart Detail""",22401217,646,1629673,"""Jordan Poole""",1610612764,"""Washington Wizards""",4,"""Pullup Jump shot""","""3PT Field Goal""","""Above the Break 3""","""Right Side Center(RC)""","""24+ ft.""",183,267,0,"""WAS""","""BOS""","""PT02M53.00S""",2024-12-16 01:10:58.200 UTC,"""REGULAR""","""Jump Shot""","""""",90,108,2024-12-16 01:17:35 UTC,6230000,false,1,"""left""",32.35,null,null,null,null,"""22401217_646""",2024-12

## Missing Data

Deal with null, NaN data

In [ ]:
# Replace nulls in pointsTotal & shotDistance with 0 - effectively what null means in this column
df_combined = df_combined.with_columns(
    pl.col('pointsTotal').fill_null(0).alias('pointsTotal')TotaT asdfasdfadsfasdfatest
)

GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,LOC_X,LOC_Y,SHOT_MADE_FLAG,HTM,VTM,clock,timeActual,periodType,subType,qualifiers,scoreHome,scoreAway,edited,orderNumber,isTargetScoreLastPeriod,isFieldGoal,side,shotDistance,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,Record_Id,game_date,atHome,time_remaining,score_margin_after,score_change,relative_margin_before,isHomeTeam
str,i64,i64,i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str,str,"datetime[μs, UTC]",str,str,str,i64,i64,"datetime[μs, UTC]",i64,bool,i64,str,f64,i64,str,i64,i64,str,datetime[μs],i32,duration[μs],i64,i32,i64,i32


Drop null values

In [ ]:
# Drop records with null values for scoreAway or ScoreHome
df_combined = df_combined.drop_nulls('score_margin_after')

df_nulls = df_combined.filter(pl.col('relative_margin_before').is_null() | pl.col('LOC_Y').is_null())
test = df_combined.filter(pl.col('relative_margin_before') == 0)

df_nulls

In [56]:
df_combined.columns

['GRID_TYPE',
 'GAME_ID',
 'GAME_EVENT_ID',
 'PLAYER_ID',
 'PLAYER_NAME',
 'TEAM_ID',
 'TEAM_NAME',
 'PERIOD',
 'ACTION_TYPE',
 'SHOT_TYPE',
 'SHOT_ZONE_BASIC',
 'SHOT_ZONE_AREA',
 'SHOT_ZONE_RANGE',
 'LOC_X',
 'LOC_Y',
 'SHOT_MADE_FLAG',
 'HTM',
 'VTM',
 'clock',
 'timeActual',
 'periodType',
 'subType',
 'qualifiers',
 'scoreHome',
 'scoreAway',
 'edited',
 'orderNumber',
 'isTargetScoreLastPeriod',
 'isFieldGoal',
 'side',
 'shotDistance',
 'pointsTotal',
 'assistPlayerNameInitial',
 'assistPersonId',
 'assistTotal',
 'Record_Id',
 'game_date',
 'atHome',
 'time_remaining',
 'score_margin_after',
 'score_change',
 'relative_margin_before',
 'isHomeTeam']

In [ ]:
# Subset of numeric columns for correlation
numerical = [
    'PERIOD',
    'LOC_X',
    'LOC_Y',
    'pointsTotal',
    'relative_margin_before',
    'shotDistance',
    'SHOT_MADE_FLAG',
    'time_remaining'
]

# Subset of columns including categorical variables only
categorical = [
    'ACTION_TYPE',
    'isHomeTeam',
    'PLAYER_ID',
    'SHOT_ZONE_BASIC',
    'SHOT_ZONE_AREA',
    'SHOT_ZONE_RANGE',
    'subType',
    'SHOT_MADE_FLAG'
]
    
# Subset of columns to be used in classifier models
classification = [   
    'PERIOD',
    'LOC_X',
    'LOC_Y',
    'pointsTotal',
    'relative_margin_before',
    'shotDistance',
    'SHOT_MADE_FLAG',
    'time_remaining'
]

df_numerical = df_combined.select(numerical)
df_categorical = df_combined.select(categorical)
df_model = df_combined.select(classification)


## Encoding

Encode categorical variables to enable use in classifiers

In [ ]:
df_encoded = df_combined.select(classification).to_dummies(
    ['subType',
     'side',
     'SHOT_ZONE_BASIC',
     'SHOT_ZONE_AREA',
     'SHOT_ZONE_RANGE'
     ]
)
df_encoded

PERIOD,LOC_X,LOC_Y,pointsTotal,relative_margin_before,shotDistance,SHOT_MADE_FLAG,subType_DUNK,subType_Hook,subType_Jump Shot,subType_Layup,side_left,side_right,time_remaining
i64,i64,i64,i64,i64,f64,i64,u8,u8,u8,u8,u8,u8,duration[μs]
1,-168,205,0,0,26.51,0,0,0,1,0,1,0,11m 43s
1,-136,-1,0,0,13.6,0,0,0,1,0,1,0,11m 38s
1,157,203,3,0,25.63,1,0,0,1,0,1,0,10m 50s
1,-176,184,0,0,25.51,0,0,0,1,0,1,0,9m 47s
1,-25,8,2,0,2.62,1,0,0,0,1,1,0,9m 44s
…,…,…,…,…,…,…,…,…,…,…,…,…,…
4,15,1,18,-17,1.48,1,0,0,0,1,1,0,3m 14s
4,183,267,0,-18,32.35,0,0,0,1,0,1,0,2m 53s
4,229,32,14,-21,23.15,1,0,0,1,0,1,0,2m 11s


In [ ]:
pddf_model = df_model.to_pandas()